# An-Ra V4 — Protected Colab T4 Trainer

This notebook continues the canonical 181M-parameter V4 model from the latest verified full-resume checkpoint. It uses one canonical writer, a signed launch contract, deterministic token windows, and Drive-backed checkpoint durability every 100 optimizer steps or 15 minutes.

**Before Run all:** select a T4 GPU runtime and ensure `MyDrive/AnRa/cluster` contains the prepared assets. Signing keys are created once in your private `MyDrive/AnRa/private` folder and are never displayed. Do not run two notebooks with `WORKER_ROLE = "canonical_trainer"` at the same time.

In [ ]:
# Operator configuration
WORKER_ROLE = "canonical_trainer"  # canonical_trainer or verify_only
WORKER_ID = "colab-t4-primary"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
REPO_REF = "iterate500"
SESSION_BUDGET_MINUTES = 180
DRAIN_RESERVE_MINUTES = 30
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8

DRIVE_ROOT = "/content/drive/MyDrive/AnRa/cluster"
BASELINE_FOLDER = f"{DRIVE_ROOT}/resume-step3-837f7721-64m"
VAULT_ROOT = f"{DRIVE_ROOT}/checkpoint-vault"
PACK_PARTS = [
    ("v4_phase_a_170m_seed1301.tar.gz.part00", 83886080, "9efe814598f52275dee15cb70e981e1bb375e24dbf97f39788aa4c84498f33f0"),
    ("v4_phase_a_170m_seed1301.tar.gz.part01", 63233323, "c073e325d2fbe09fe4afefe75c251db59540ea3358e34d8a184d7bb2831e0f6a"),
]
PACK_ARCHIVE_SHA256 = "07f01bf4809667acc670eb9c94dfab38d28522d7bfc2d4c930e71898cff86ee7"
print({"role": WORKER_ROLE, "worker": WORKER_ID, "training_minutes": SESSION_BUDGET_MINUTES - DRAIN_RESERVE_MINUTES})

In [ ]:
# Mount the authorized account's Drive and enforce the requested accelerator.
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, torch
assert torch.cuda.is_available(), "No CUDA GPU. Select Runtime > Change runtime type > T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert "T4" in gpu_name.upper(), f"Expected a T4 runtime, received {gpu_name}"
assert total_gib >= 14, f"T4 memory contract failed: {total_gib:.1f} GiB"
assert os.path.isdir(DRIVE_ROOT), f"Missing shared Drive folder: {DRIVE_ROOT}"
subprocess.run(["nvidia-smi"], check=True)
print(f"READY: {gpu_name}, {total_gib:.1f} GiB")

In [ ]:
# Clone a clean operational checkout. The signed launch records the exact commit.
import pathlib, shutil, subprocess
REPO = pathlib.Path('/content/anra')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--filter=blob:none", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert not subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", "."], check=True)
print(f"Clean source commit: {commit}")

In [ ]:
# Reconstruct and verify the immutable 170M-token V4 data window locally.
import hashlib, tarfile

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

SCRATCH = pathlib.Path('/content/anra-scratch')
SCRATCH.mkdir(parents=True, exist_ok=True)
archive = SCRATCH / 'v4_phase_a_170m_seed1301.tar.gz'
temporary = archive.with_suffix(archive.suffix + '.tmp')
with temporary.open('wb') as target:
    for name, expected_size, expected_hash in PACK_PARTS:
        part = pathlib.Path(DRIVE_ROOT) / name
        assert part.is_file(), f"Missing data pack part: {part}"
        assert part.stat().st_size == expected_size, f"Wrong size: {part}"
        assert sha256_file(part) == expected_hash, f"Corrupt data pack part: {part}"
        with part.open('rb') as source:
            shutil.copyfileobj(source, target, 8 * 1024 * 1024)
temporary.replace(archive)
assert sha256_file(archive) == PACK_ARCHIVE_SHA256, "Data archive hash mismatch"
pack_parent = REPO / 'output' / 'v2' / 'cloud_packs'
pack_parent.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall(pack_parent, filter='data')
PACK_ROOT = pack_parent / 'v4_phase_a_170m_seed1301'
assert (PACK_ROOT / 'pack_manifest.json').is_file()
print(f"Verified data pack: {archive.stat().st_size:,} compressed bytes")

In [ ]:
# Materialize the newest full-resume checkpoint: vault first, prepared baseline otherwise.
import json

vault_pointer = pathlib.Path(VAULT_ROOT) / 'canonical.json'
if vault_pointer.is_file():
    pointer = json.loads(vault_pointer.read_text())
    manifest_path = pathlib.Path(VAULT_ROOT) / 'manifests' / f"{pointer['snapshot_id']}.json"
    chunk_root = pathlib.Path(VAULT_ROOT) / 'chunks'
    nested_chunks = True
    source_label = 'latest canonical Drive vault checkpoint'
else:
    manifest_path = pathlib.Path(BASELINE_FOLDER) / 'manifest.json'
    chunk_root = pathlib.Path(BASELINE_FOLDER)
    nested_chunks = False
    source_label = 'prepared step-3 baseline checkpoint'

manifest = json.loads(manifest_path.read_text())
assert manifest['artifact_class'] == 'full_resume'
assert manifest.get('resume_eligible') is True
resume_checkpoint = SCRATCH / 'resume-source.pt'
if not (resume_checkpoint.is_file() and resume_checkpoint.stat().st_size == manifest['source']['size_bytes'] and sha256_file(resume_checkpoint) == manifest['source']['sha256']):
    temporary = resume_checkpoint.with_suffix('.pt.tmp')
    digest = hashlib.sha256()
    with temporary.open('wb') as target:
        for expected_index, record in enumerate(manifest['chunks']):
            assert record['index'] == expected_index
            name = f"{record['sha256']}.chunk"
            chunk = chunk_root / record['sha256'][:2] / name if nested_chunks else chunk_root / name
            assert chunk.is_file(), f"Missing checkpoint chunk: {chunk}"
            assert chunk.stat().st_size == record['size_bytes'], f"Wrong chunk size: {chunk}"
            assert sha256_file(chunk) == record['sha256'], f"Corrupt checkpoint chunk: {chunk}"
            with chunk.open('rb') as source:
                while block := source.read(8 * 1024 * 1024):
                    target.write(block)
                    digest.update(block)
    temporary.replace(resume_checkpoint)
assert resume_checkpoint.stat().st_size == manifest['source']['size_bytes']
verified_checkpoint_hash = sha256_file(resume_checkpoint)
assert verified_checkpoint_hash == manifest['source']['sha256']
print(f"Verified {source_label}: step={manifest['lineage']['progress']['global_step']} sha256={verified_checkpoint_hash}")

In [ ]:
# Load or create owner-private signing keys. They never enter Git or the shared cluster folder.
import secrets
private_dir = pathlib.Path('/content/drive/MyDrive/AnRa/private')
private_dir.mkdir(parents=True, exist_ok=True)
key_file = private_dir / 'training-signing-keys.json'
if key_file.is_file():
    private_keys = json.loads(key_file.read_text())
else:
    private_keys = {
        'manifest': secrets.token_hex(32),
        'evidence': secrets.token_hex(32),
    }
    key_tmp = key_file.with_suffix('.tmp')
    key_tmp.write_text(json.dumps(private_keys))
    key_tmp.replace(key_file)
manifest_key = str(private_keys.get('manifest', ''))
evidence_key = str(private_keys.get('evidence', ''))
assert len(manifest_key) >= 64 and len(evidence_key) >= 64
os.environ['ANRA_MANIFEST_SIGNING_KEY'] = manifest_key
os.environ['ANRA_EVIDENCE_SIGNING_KEY'] = evidence_key
os.environ['ANRA_REQUIRE_SIGNED_EVIDENCE'] = '1'
print('Owner-private signing keys loaded without disclosure.')

In [ ]:
# Create and validate a launch bound to this commit, checkpoint, tokenizer, and remaining token window.
launch = REPO / 'output' / 'v2' / 'launch_manifests' / f'{WORKER_ID}.json'
artifact = SCRATCH / f'anra-v4-{WORKER_ID}.pt'
create_command = [
    'python', '-m', 'scripts.create_cloud_launch',
    '--pack-root', str(PACK_ROOT),
    '--output', str(launch),
    '--artifact-path', str(artifact),
    '--checkpoint-source', str(resume_checkpoint),
    '--worker-id', WORKER_ID,
    '--runtime-estimate-hours', str(SESSION_BUDGET_MINUTES / 60),
    '--batch-size', str(BATCH_SIZE),
    '--accumulation', str(GRADIENT_ACCUMULATION),
]
subprocess.run(create_command, check=True)
signed = json.loads(launch.read_text())
assert signed['git_commit'] == commit
print({
    'run_id': signed['run_id'],
    'commit': signed['git_commit'],
    'window': signed['token_window'],
    'checkpoint': signed['checkpoint_source_hash'],
})

In [ ]:
# Start the only canonical writer. New full-resume states are protected in Drive while training continues.
if WORKER_ROLE == 'verify_only':
    print('Verification complete. This worker will not modify canonical weights.')
else:
    assert WORKER_ROLE == 'canonical_trainer'
    pathlib.Path(VAULT_ROOT).mkdir(parents=True, exist_ok=True)
    os.environ['ANRA_DURABILITY_OUTBOX'] = str(SCRATCH / 'durability-outbox')
    os.environ['ANRA_DURABILITY_REPLICAS'] = json.dumps([
        {'name': 'drive-vault', 'path': VAULT_ROOT, 'kind': 'mounted_drive', 'canonical': True}
    ])
    os.environ['ANRA_DURABILITY_MIN_PROTECTED_REPLICAS'] = '1'
    os.environ['ANRA_DURABILITY_COPY_STREAMS'] = '2'
    os.environ['ANRA_DURABILITY_ACK_TIMEOUT_SECONDS'] = '1800'
    os.environ['ANRA_CHECKPOINT_EVERY_MIN'] = '15'
    os.environ['ANRA_DURABLE_CHECKPOINT_STEPS'] = '100'
    train_command = [
        'python', '-u', '-m', 'training.train_unified',
        '--mode', 'session',
        '--launch-manifest', str(launch),
        '--prepare_data', 'never',
        '--post-session-eval', 'none',
        '--data_path', 'training_data/anra_training.txt',
    ]
    subprocess.run(train_command, check=True, env=os.environ.copy())
    print('TRAINING SESSION COMPLETE. The final protected checkpoint is in checkpoint-vault.')